# This notebook generates the initial model for 1-D inversion.

The initial model is a 1-D layered model. At each depth, the velocity is the horizontal average of the Crust 1.0 model.

In [ ]:
# Import PyTomoATT
# Documentation: https://tomoatt.github.io/PyTomoATT/index.html

import pytomoatt as pyt

import numpy as np

In [ ]:
# # Import TomoATT data-processing utilities
# import sys
# sys.path.append('../utils')
# import functions_for_data as ffd

In [ ]:
import numpy as np
print(np.__version__)
# PyTomoATT may fail when NumPy version is <= 1.26.4.

# 1. Create a PyTomoATT Model Object

In [ ]:
# TomoATT parameter file, mainly used to read the study-region settings
par_file = "../3_input_params/input_params_step1_1D_inv.yaml"

# Read the parameter file and create an ATTModel object
att_model = pyt.ATTModel(par_file)

# The 3-D grid information of this model includes
n_rtp          = att_model.n_rtp       # number of grid nodes in three directions r (depth), t (longtitude), p (latitude) directions
am_depths      = att_model.depths      # depth in the r direction (km)
am_latitudes   = att_model.latitudes   # latitude in the p direction (degree)
am_longitudes  = att_model.longitudes  # longitude in the t direction (degree)

# Print grid-coordinate information
# print("grid node numbers (N_r, N_t, N_p):", n_rtp)
# print("depths (km):", am_depths)
# print("latitudes (degree):", am_latitudes)
# print("longitudes (degree):", am_longitudes)

In [ ]:
# Region-rotation parameters
central_lat = 37.0      # rotation-center latitude
central_lon = 37.0      # rotation-center longitude
rotation_angle = 45.0   # clockwise rotation angle

# Generate the Crust 1.0 model
# Note: set rotate to None if no rotation is needed.
rotate = [central_lat, central_lon, rotation_angle]
att_model.grid_data_crust1(rotate=rotate)

# 2. Generate an Averaged 1-D Model (optional)

The initial inversion model can be a 1-D model. A common method is horizontal averaging, which is demonstrated here.

The initial model can also be a 3-D model. A common approach is to interpolate an existing model into the study region and then apply suitable smoothing, such as Gaussian smoothing, to reduce the risk of convergence to local minima.

In [ ]:
# Obtain the 3-D model array
vel_3d_array = att_model.vel

# Compute the horizontal average as the 1-D model
vel_1d_array = np.mean(vel_3d_array, axis=(1,2))

# Broadcast the 1-D model to the 3-D grid
att_model.vel = np.broadcast_to(vel_1d_array[:, np.newaxis, np.newaxis], vel_3d_array.shape)

In [ ]:
# Write to file
import os
output_path = "./output_model"
os.makedirs(output_path, exist_ok=True)

fname = "%s/model_1d_crust1.0_N%d_%d_%d.h5" % (output_path, n_rtp[0], n_rtp[1], n_rtp[2])
att_model.write(fname)


# Back up a copy to the main directory
output_path = "../2_models"
os.makedirs(output_path, exist_ok=True)
fname = "%s/model_1d_crust1.0_N%d_%d_%d.h5" % (output_path, n_rtp[0], n_rtp[1], n_rtp[2])
att_model.write(fname)

# 3. Plot the 1-D Model (optional)

In [ ]:
att_model_1d = pyt.ATTModel.read(fname,par_file)

all_dep = np.array(att_model_1d.depths)

vel_1d = np.array(att_model_1d.vel[:, 0, 0])


# Plot the layered model
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(6,6))
gridspace = GridSpec(6,6,figure = fig)
ax2 = fig.add_subplot(gridspace[0:6, 0:6])


ax2.plot(vel_1d,all_dep,'r-',label="1d initial")
ax2.legend(fontsize = 14)
ax2.tick_params(axis='x',labelsize=18)
ax2.tick_params(axis='y',labelsize=18)
ax2.set_ylabel('Depth (km)',fontsize=18)
ax2.set_xlabel('P wave velocity (km/s)',fontsize=18)
ax2.set_ylim([0,50])
ax2.set_xlim([4.5,8.2])
ax2.invert_yaxis()
ax2.set_title('1D model inversion',fontsize=18)
ax2.grid()

os.makedirs("figs", exist_ok=True)
fig.savefig("figs/crust1p0_1d.png",dpi=300,bbox_inches='tight',facecolor='w',edgecolor='w')